> **图 3.1**：对于例 3.2，预测值 $\widehat{Y}_0$（蓝色实线）和 $\widehat{Y}_0^\mathrm{L}$（红色实线）关于 $y_1\in(0,1)$

In [1]:
if(!dir.exists("data"))dir.create("data")
if(!file.exists("data/3.1-plot.RData")){
  y1=seq(0,1,length.out=200)
  y.ce=y1^2/2
  y.blup=-1/12+0.5*y1
  save(y1,y.ce,y.blup,file="data/3.1-plot.RData")
}
load("data/3.1-plot.RData")
library(svglite)
svglite("../markdown/figures/3.1.svg",width=4,height=4)
plot(y1,y.ce,type="l",xlab="y1",ylab="y0 预测量",xlim=c(0,1))
lines(y1,y.blup,lty=2)
dev.off()


agg_record_4e843b46663e 
                      2

> **图 3.2**：示例 1.1 所用 $40$ 个输入点的散点图矩阵

In [6]:
load("data/1.1-plot.RData")
library(svglite)
svglite("../markdown/figures/3.2.svg",width=8,height=8)
pairs(df[,c("A","H","F","LC")],pch=16,labels=c("房间面积","房间高度","火焰高度","损失比例"))
dev.off()

agg_record_655412942ab8 
                      2

> **图 3.3**：利用例 1.1 的数据绘制火灾蔓延至火源上方 $5$ 英尺所需时间分别与各输入参数的散点图：（1）热损失系数（2）火源距地面高度（3）房间高度（4）房间面积

In [1]:
load("data/1.1-plot.RData")
library(svglite)
svglite("../markdown/figures/3.3.svg",width=8,height=8)
par(mfrow=c(2,2),mar=c(4,4,1,1))
plot(df$LC,df$y,pch=16,cex=0.7,xlab="热损失系数",ylab="到达5英尺所需时间（秒）")
plot(df$F,df$y,pch=16,cex=0.7,xlab="火源距地面高度（英尺）",ylab="到达5英尺所需时间（秒）")
plot(df$H,df$y,pch=16,cex=0.7,xlab="房间高度（英尺）",ylab="到达5英尺所需时间（秒）")
plot(df$A,df$y,pch=16,cex=0.7,xlab="房间面积（平方英尺）",ylab="到达5英尺所需时间（秒）")
par(mfrow=c(1,1))
dev.off()


agg_record_19c478367f19 
                      2

> **图 3.4**：示例 1.1 中使用的 $320$ 个等距网格点，火源上方 $5$ 英尺处真实到达时间与预测到达时间的散点图

In [3]:
if(!dir.exists("data"))dir.create("data")
if(!file.exists("data/3.4-plot.RData")){
  load("data/1.1-plot.RData")
  CT=1;CL=1;G=32.2;CP=0.24;PA=530.0;DA=0.075;LR=0.35;Q0=0.1;targetZ=5
  QA_peak=1000;tpk=180;TMAX=1200
  QAt=function(t)if(t<=tpk)QA_peak*t/tpk else QA_peak
  QTfun=function(t)QAt(t)/Q0
  X=as.matrix(df[,c("LC","F","H","A")]);y=df$y;n=nrow(X);one=rep(1,n)
  nugget=1e-4
  neg_REML=function(lnxi){
    xi=exp(lnxi)
    D2=matrix(0,n,n)
    for(j in 1:4)D2=D2+outer(X[,j],X[,j],function(a,b)xi[j]*(a-b)^2)
    Rm=exp(-D2)+diag(n)*nugget
    C=chol(Rm);Rinv=chol2inv(C)
    beta=as.numeric((t(one)%*%Rinv%*%y)/(t(one)%*%Rinv%*%one))
    resid=y-one*beta
    s2=as.numeric(t(resid)%*%Rinv%*%resid)/(n-1)
    -(-sum(log(diag(C)))-0.5*(n-1)*log(s2)-0.5*log(as.numeric(t(one)%*%Rinv%*%one)))
  }
  fit=optim(log(c(1,0.05,0.01,0.0001)),neg_REML,method="L-BFGS-B",lower=log(1e-7),upper=log(100),
            control=list(maxit=5000,factr=1e-12))
  xi=exp(fit$par)
  D2=matrix(0,n,n)
  for(j in 1:4)D2=D2+outer(X[,j],X[,j],function(a,b)xi[j]*(a-b)^2)
  Rm=exp(-D2)+diag(n)*nugget
  Rinv=chol2inv(chol(Rm))
  beta0=as.numeric((t(one)%*%Rinv%*%y)/(t(one)%*%Rinv%*%one))
  resid=y-one*beta0
  s2=as.numeric(t(resid)%*%Rinv%*%resid)/(n-1)
  krig=function(x0){
    r=exp(-rowSums(sweep(X,2,x0)^2*rep(xi,each=n)))
    beta0+as.numeric(t(r)%*%Rinv%*%(y-one*beta0))
  }
  lc=seq(0.6,0.9,length.out=4);h=seq(8,12,length.out=4);f=seq(1,3,length.out=4);a=seq(81,256,length.out=5)
  grid=expand.grid(LC=lc,H=h,F=f,A=a)
  ytrue=apply(grid,1,function(r)asetb_orig(A=r["A"],H=r["H"],F=r["F"],LC=r["LC"]))
  ypred=apply(grid,1,function(r)krig(as.numeric(r[c("LC","F","H","A")])))
  save(xi,beta0,s2,ytrue,ypred,file="data/3.4-plot.RData")
}
load("data/3.4-plot.RData")
library(svglite)
svglite("../markdown/figures/3.4.svg",width=4,height=4)
par(mar=c(4,4,1,1))
plot(ytrue,ypred,pch=16,cex=0.7,xlab="真实到达时间（秒）",ylab="预测到达时间（秒）")
abline(0,1,lty=2)
dev.off()


agg_record_19c46d2e34db 
                      2